# 🏎️ F1 World Champion Predictor
## AI Project — Deliverable 2
**Roll Numbers:** 24L-0679 | 24L-0912 | 24L-0518

---
This notebook predicts whether a Formula 1 driver will become a **World Champion**
using Multiple Linear Regression built from scratch in Python.

## 1. Import Libraries

In [ ]:
import csv
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, mean_absolute_error
)

print("All libraries imported successfully!")

## 2. Load Dataset

In [ ]:
df = pd.read_csv('F1DriversDataset.csv')
print(f"Dataset shape: {df.shape}")
df.head()

## 3. Data Description

In [ ]:
print("=== Dataset Info ===")
df.info()

In [ ]:
print("=== Statistical Summary ===")
df.describe()

In [ ]:
print("=== Champion Distribution ===")
print(df['Champion'].value_counts())
print(f"\nChampions   : {df['Champion'].sum()}")
print(f"Non-Champions: {(~df['Champion']).sum()}")

## 4. Data Visualization

In [ ]:
# --- Champion vs Non-Champion Count ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
counts = df['Champion'].value_counts()
axes[0].bar(['Non-Champion', 'Champion'], counts.values, color=['#4C72B0', '#DD8452'])
axes[0].set_title('Champion vs Non-Champion Count')
axes[0].set_ylabel('Number of Drivers')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 2, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(counts.values, labels=['Non-Champion', 'Champion'],
            autopct='%1.1f%%', colors=['#4C72B0', '#DD8452'], startangle=90)
axes[1].set_title('Champion Proportion')

plt.tight_layout()
plt.savefig('champion_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved!")

In [ ]:
# --- Race Wins Distribution ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

champions = df[df['Champion'] == True]['Race_Wins']
non_champs = df[df['Champion'] == False]['Race_Wins']

axes[0].hist(non_champs, bins=30, color='#4C72B0', alpha=0.7, label='Non-Champion')
axes[0].hist(champions, bins=15, color='#DD8452', alpha=0.9, label='Champion')
axes[0].set_title('Race Wins Distribution')
axes[0].set_xlabel('Race Wins')
axes[0].set_ylabel('Number of Drivers')
axes[0].legend()

# Points comparison
axes[1].hist(df[df['Champion']==False]['Points'], bins=30, color='#4C72B0', alpha=0.7, label='Non-Champion')
axes[1].hist(df[df['Champion']==True]['Points'],  bins=15, color='#DD8452', alpha=0.9, label='Champion')
axes[1].set_title('Career Points Distribution')
axes[1].set_xlabel('Points')
axes[1].legend()

plt.tight_layout()
plt.savefig('distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Correlation Heatmap ---
feature_cols = ['Race_Entries','Race_Wins','Pole_Positions','Podiums',
                'Fastest_Laps','Points','Win_Rate','Podium_Rate','FastLap_Rate','Points_Per_Entry']

corr_df = df[feature_cols].copy()
corr_df['Champion'] = df['Champion'].astype(int)

plt.figure(figsize=(10, 7))
sns.heatmap(corr_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Top 10 drivers by wins ---
top10 = df.nlargest(10, 'Race_Wins')[['Driver','Race_Wins','Champion']]
colors_bar = ['#DD8452' if c else '#4C72B0' for c in top10['Champion']]

plt.figure(figsize=(10, 5))
bars = plt.barh(top10['Driver'], top10['Race_Wins'], color=colors_bar)
plt.xlabel('Race Wins')
plt.title('Top 10 Drivers by Race Wins  (Orange = Champion)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('top10_wins.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Data Preprocessing

In [ ]:
# Step 1: Drop irrelevant columns
drop_cols = ['Seasons', 'Championship Years', 'Decade', 'Active', 'Start_Rate', 'Nationality']
df_clean = df.drop(columns=[c for c in drop_cols if c in df.columns])
print(f"Step 1 — Dropped irrelevant columns. Remaining columns: {list(df_clean.columns)}")

In [ ]:
# Step 2: Remove non-racing records (Race_Entries or Race_Starts == 0)
before = len(df_clean)
df_clean = df_clean[(df_clean['Race_Entries'] > 0) & (df_clean['Race_Starts'] > 0)]
print(f"Step 2 — Removed {before - len(df_clean)} non-racing records. Remaining: {len(df_clean)}")

In [ ]:
# Step 3: Check for duplicates
dupes = df_clean.duplicated(subset='Driver').sum()
print(f"Step 3 — Duplicate driver entries found: {dupes}")

In [ ]:
# Step 4: Check missing values
print("Step 4 — Missing values per column:")
print(df_clean[feature_cols].isnull().sum())

In [ ]:
# Step 5: Min-Max Normalisation
def minmax_normalize(df, cols):
    df_norm = df.copy()
    mins, maxs = {}, {}
    for col in cols:
        mins[col] = df[col].min()
        maxs[col] = df[col].max()
        denom = maxs[col] - mins[col]
        df_norm[col] = (df[col] - mins[col]) / denom if denom != 0 else 0
    return df_norm, mins, maxs

df_norm, mins, maxs = minmax_normalize(df_clean, feature_cols)
print("Step 5 — Min-Max Normalisation applied.")
print(df_norm[feature_cols].describe().round(3))

## 6. Train / Test Split (80/20)

In [ ]:
from sklearn.model_selection import train_test_split as sk_split

X = df_norm[feature_cols].values.tolist()
y = [1.0 if v else 0.0 for v in df_norm['Champion'].values]

# Convert to lists for our scratch model
combined = list(zip(X, y))
random.seed(42)
random.shuffle(combined)

split = int(len(combined) * 0.8)
train_data = combined[:split]
test_data  = combined[split:]

X_train = [row[0] for row in train_data]
y_train = [row[1] for row in train_data]
X_test  = [row[0] for row in test_data]
y_test  = [row[1] for row in test_data]

print(f"Training set : {len(X_train)} drivers")
print(f"Test set     : {len(X_test)} drivers")

## 7. Machine Learning Model — Multiple Linear Regression (from scratch)

In [ ]:
def predict(features, weights, bias):
    return bias + sum(w * x for w, x in zip(weights, features))

def train_model(X_train, y_train, lr=0.01, epochs=10000):
    n = len(X_train)
    num_features = len(X_train[0])
    weights = [0.0] * num_features
    bias = 0.0
    loss_history = []

    for epoch in range(1, epochs + 1):
        w_grad = [0.0] * num_features
        b_grad = 0.0

        for features, label in zip(X_train, y_train):
            error = predict(features, weights, bias) - label
            for i, x in enumerate(features):
                w_grad[i] += error * x
            b_grad += error

        for i in range(num_features):
            weights[i] -= lr * (w_grad[i] / n)
        bias -= lr * (b_grad / n)

        if epoch % 500 == 0 or epoch == 1:
            mse = sum((predict(f, weights, bias) - l)**2 for f, l in zip(X_train, y_train)) / n
            loss_history.append((epoch, mse))

    return weights, bias, loss_history

weights, bias, loss_history = train_model(X_train, y_train)
print("Model training complete!")
print(f"Final weights: {[round(w,4) for w in weights]}")
print(f"Bias: {round(bias, 4)}")

In [ ]:
# Plot training loss curve
epochs_plot = [x[0] for x in loss_history]
mse_plot    = [x[1] for x in loss_history]

plt.figure(figsize=(8, 4))
plt.plot(epochs_plot, mse_plot, color='#DD8452', linewidth=2)
plt.title('Training Loss (MSE) over Epochs')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('training_loss.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Model Evaluation

In [ ]:
# Get raw predictions
raw_preds = [predict(f, weights, bias) for f in X_test]

# Classify: >= 0.5 → Champion
y_pred = [1 if p >= 0.5 else 0 for p in raw_preds]
y_true = [int(v) for v in y_test]

# ── Regression Metrics ──
n = len(y_true)
mse  = sum((p - a)**2 for p, a in zip(raw_preds, y_true)) / n
rmse = math.sqrt(mse)
mae  = sum(abs(p - a) for p, a in zip(raw_preds, y_true)) / n
mean_actual = sum(y_true) / n
ss_tot = sum((a - mean_actual)**2 for a in y_true)
ss_res = sum((a - p)**2 for a, p in zip(y_true, raw_preds))
r2 = 1 - (ss_res / ss_tot) if ss_tot != 0 else 0

# ── Classification Metrics ──
acc       = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall    = recall_score(y_true, y_pred, zero_division=0)
f1        = f1_score(y_true, y_pred, zero_division=0)
cm        = confusion_matrix(y_true, y_pred)

print("=" * 45)
print("       REGRESSION METRICS")
print("=" * 45)
print(f"  MAE  : {mae:.6f}")
print(f"  MSE  : {mse:.6f}")
print(f"  RMSE : {rmse:.6f}")
print(f"  R²   : {r2:.6f}")

print("\n" + "=" * 45)
print("       CLASSIFICATION METRICS")
print("=" * 45)
print(f"  Accuracy  : {acc*100:.2f}%")
print(f"  Precision : {precision:.4f}")
print(f"  Recall    : {recall:.4f}")
print(f"  F1 Score  : {f1:.4f}")
print("\nConfusion Matrix:")
print(cm)

In [ ]:
# Plot Confusion Matrix
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Non-Champion','Champion'],
            yticklabels=['Non-Champion','Champion'])
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Summary metrics bar chart
metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
values  = [acc, precision, recall, f1]

plt.figure(figsize=(7, 4))
bars = plt.bar(metrics, values, color=['#4C72B0','#DD8452','#55A868','#C44E52'])
plt.ylim(0, 1.1)
plt.title('Classification Metrics Summary')
plt.ylabel('Score')
for bar, val in zip(bars, values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{val:.3f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('metrics_summary.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Summary
- Model trained using **Multiple Linear Regression with Batch Gradient Descent** from scratch
- Achieved high accuracy on unseen test data
- All regression and classification metrics computed and visualized